# REST API demo

Start the API in a terminal with `python app.py`, then run the cells below.

In [1]:
from urllib.request import urlopen
import json

with urlopen('http://127.0.0.1:8000/health') as response:
    health = json.load(response)
health

{'status': 'ok', 'service': 'GiftFlow API'}

In [ ]:
with urlopen('http://127.0.0.1:8000/hello') as response:
    greeting = json.load(response)
greeting

# Type Hints and Optional
  
   type hints is nothing but it tells python what type of data input and must be in expected contain
   optional used for if accpet both datatype then use it like int|str or optional[int]=str
   
   

In [25]:
from typing import Optional,Union,TypedDict,Literal,Callable
OrderStatus = Literal["pending", "shipped", "delivered", "cancelled"]
class Product(TypedDict):
    product_id:int
    product_name:str
    total_amount:float
    price:Optional[Union[float, int]]
    available:bool
    discount:Optional[float]
    order_status:OrderStatus
def cal_tot_amt(price:Optional[Union[float, int]], discount:Optional[float])->float:
    if price is None:
        return 0.0
    if discount is None:
        return price
    return float(price * (1 - discount))
def product_details(
        product_id:int,
        product_name:str,
        total_amount:Callable[
                    [Optional[Union[float, int]], Optional[float]],
                    float
                    ],
        price:Optional[Union[float, int]]=None,
        available:bool = True,
        discount:Optional[float]=None,  
        order_status:Optional[OrderStatus]=None
)->Product:
    if order_status is None:
        order_status = "pending"
    return {
        "product_id": product_id,
        "product_name": product_name,
        "price": price,
        "available": available,
        "discount": discount,
        "total_amount": total_amount(price, discount),
        "order_status": order_status
    }

details:Product = product_details(
    product_id=1,
    product_name="Widget",
    total_amount=cal_tot_amt,
    price=19.99,        
    available=True,
    discount=0.1,
  
    order_status="shipped"
)

for key, value in details.items():
    print(f"{key}: {value}")
product_catalog: list[Product] = [
    {
        "product_id": 1,
        "product_name": "Air Conditioner",
        "price": 299.99,
        "available": True,
        "discount"  : 0.15,
        "order_status": "pending"
    },
    {
        "product_id": 2,
        "product_name": "Android Phone",      
        "available": True,
        "discount": 0.1,
        "order_status":"cancelled"
    },
    {
        "product_id": 3,
        "product_name": "Laptop",        
        "price": 999.99,
        "available": False,
        "discount": None,
        "order_status": "delivered"
    }
]

def total_number_of_products(catalog:list[Product]) -> int:
    return len(catalog)
total_products = total_number_of_products(product_catalog)
print(f"Total number of products: {total_products}")

def get_product_by_id(catalog:list[Product],product_id:int)->Optional[Product]:
    for product in catalog:
        if product["product_id"] == product_id:
            return product
    return None
product = get_product_by_id(product_catalog, 2)



def get_all(catalog:list[Product])->list:
    print("PRODUCT ID | PRODUCT NAME | PRICE | AVAILABLE | DISCOUNT | ORDER STATUS")
    for product in catalog:
       print()
       for value in product.values():
            print(f"{value} |",end =" ")
            
print(get_all(product_catalog))



product_id: 1
product_name: Widget
price: 19.99
available: True
discount: 0.1
total_amount: 17.991
order_status: shipped
Total number of products: 3
PRODUCT ID | PRODUCT NAME | PRICE | AVAILABLE | DISCOUNT | ORDER STATUS

1 | Air Conditioner | 299.99 | True | 0.15 | pending | 
2 | Android Phone | True | 0.1 | cancelled | 
3 | Laptop | 999.99 | False | None | delivered | None


In [4]:
def user_details(*salary:int,**details:dict)->list:
    for key,value in details.items():
        print(f"{key}: {value}")
    print(f"salary:{float(salary[0])}")
details = user_details(50000, name = "John Doe", email = "john.doe@example.com", age = 30, country = "USA")
print(details)

name: John Doe
email: john.doe@example.com
age: 30
country: USA
salary:50000.0
None


In [ ]:
from pydantic import BaseModel, Field,EmailStr,field_validator
from typing import Optional
from fastapi import FastAPI,Depends,HTTPException
import asyncio

user_data= {
   1: {
        "id":1,
        "name": "John Doe",
        "email": "john.doe@example.com",
        "phone": "1234567890"
    }
}

class UserBase(BaseModel):

    name: str 
    email: EmailStr
    phone: str

class User(UserBase):
    id: Optional[int] = None

class UserRequest(UserBase):


    @field_validator("name")
    @classmethod
    def validate_name(cls,value):
        if len(value)<1 and len(value)>10:
            raise ValueError("Name must be between 1 and 10 characters")
        return value
    
    @field_validator("phone")
    @classmethod
    def validate_phone(cls,value,mode = "plain"):
        if isinstance(value,str) and len(value)<1 and len(value)>12:
            value = int(value)
        return value
    
class UserResponse(BaseModel):
    message: str
    user:Optional[User]=None

def signup(user_request: UserRequest,db:dict) -> UserResponse:

    for user in db.values():
        if user["email"] == user_request.email:
            return UserResponse(
                message="User with this email already exists",
                user=None
            )
    user_id = max(db.keys(), default=0) + 1
    new_user = User(id=user_id, **user_request.dict())
    db[user_id] = new_user.model_dump()
    return UserResponse(
        message="User account created successfully",
        user=new_user
    )
def login(user_id:int,db:dict)->UserResponse:

    user = db.get(user_id)
    if user is None:
        raise HTTPException(
            status_code=404,
            detail="User not found"
            )
    
    return UserResponse(
        message="User logged in successfully",
        user=user
    )
def get_db():

    return user_data

app = FastAPI(title="User Management API", description="API for user signup and login", version="1.0.0")

@app.get("/health", summary="Health Check", description="Check the health of the API")
def health_check():

    return {"status": "healthy"}

@app.post('/signup', summary="User Signup", description="Create a new user account")
def signup_endpoint(user_request: UserRequest, db: dict = Depends(get_db))-> UserResponse:

    user_response = signup(user_request,db)
    return user_response

@app.post('/login', summary="User Login", description="Login with user ID and details")
def login_endpoint(user_id: int,db:dict = Depends(get_db))-> UserResponse:

    try:
        user_response = login(user_id,db)
        return user_response
    except HTTPException as e:
        raise e
    
if __name__ == "__main__":

    import uvicorn
    config = uvicorn.Config(app, host="127.0.0.1", port=8001)
    server = uvicorn.Server(config)
    server_task = asyncio.create_task(server.serve())

    for key, value in user_data.items():
        print(f"{key}: {value}")

1: {'id': 1, 'name': 'John Doe', 'email': 'john.doe@example.com', 'phone': '1234567890'}


INFO:     Started server process [4476]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://127.0.0.1:8001 (Press CTRL+C to quit)


INFO:     127.0.0.1:53074 - "GET / HTTP/1.1" 404 Not Found
INFO:     127.0.0.1:53074 - "GET /docs HTTP/1.1" 200 OK
INFO:     127.0.0.1:53074 - "GET /openapi.json HTTP/1.1" 200 OK
INFO:     127.0.0.1:53080 - "POST /login?user_id=1 HTTP/1.1" 200 OK
